In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [54]:
df = pd.read_csv('diabetes.csv')
print("Датасет успешно загружен!")
print(f"Размер датасета: {df.shape}")
df.head()

Датасет успешно загружен!
Размер датасета: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [73]:
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_zeros:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

X = df.drop(columns=['Outcome'])
y = df['Outcome']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("Балансировка выполнена с помощью SMOTE.")

from sklearn.utils import resample
df_train = pd.concat([X_train, y_train], axis=1)
df_majority = df_train[df_train.Outcome == 0]
df_minority = df_train[df_train.Outcome == 1]
df_minority_upsampled = resample(df_minority, replace=True, n_samples=len(df_majority), random_state=42)
df_resampled = pd.concat([df_majority, df_minority_upsampled])
X_train_resampled = df_resampled.drop(columns=['Outcome'])
y_train_resampled = df_resampled['Outcome']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13],
    'weights': ['uniform', 'distance']
}

knn_grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

knn_grid.fit(X_train_scaled, y_train_resampled)

# 8. Вывод лучших результатов
print("\n=== Обучение k-NN завершено ===")
print(f"Лучшие параметры: {knn_grid.best_params_}")
print(f"Лучший результат ROC-AUC на кросс-валидации: {knn_grid.best_score_:.4f}")

Балансировка выполнена с помощью SMOTE.

=== Обучение k-NN завершено ===
Лучшие параметры: {'n_neighbors': 13, 'weights': 'distance'}
Лучший результат ROC-AUC на кросс-валидации: 0.9617
